In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
import glob

print("1. ADIM: data/ klasöründeki tüm CSV dosyaları okunuyor...")
# Yol '../data/' olarak güncellendi
csv_dosyalari = glob.glob("../data/*.csv")

df_listesi = []
for dosya in csv_dosyalari:
    temp_df = pd.read_csv(dosya, usecols=['DATE_TIME', 'NUMBER_OF_VEHICLES'])
    temp_df.columns = temp_df.columns.str.strip().str.upper()
    temp_df = temp_df.dropna(subset=['NUMBER_OF_VEHICLES'])
    temp_df['DATE_TIME'] = pd.to_datetime(temp_df['DATE_TIME'])
    
    gruplanmis = temp_df.groupby('DATE_TIME')['NUMBER_OF_VEHICLES'].sum().reset_index()
    gruplanmis = gruplanmis[gruplanmis['NUMBER_OF_VEHICLES'] > 50000]
    
    if not gruplanmis.empty:
        df_listesi.append(gruplanmis)

df_genel = pd.concat(df_listesi, ignore_index=True)
df_genel = df_genel.sort_values(by='DATE_TIME').reset_index(drop=True)

print(f"\n[BİLGİ] Filtrelemeden sonra toplam temiz satır sayısı: {len(df_genel)}")

print("\n2. ADIM: Özellikler çıkarılıyor...")
df_genel['saat'] = df_genel['DATE_TIME'].dt.hour
df_genel['gun_kod'] = df_genel['DATE_TIME'].dt.dayofweek
df_genel['hafta_sonu'] = df_genel['gun_kod'].apply(lambda x: 1 if x >= 5 else 0)
df_genel['bir_saat_once'] = df_genel['NUMBER_OF_VEHICLES'].shift(1)
df_genel = df_genel.dropna().reset_index(drop=True)

min_arac = float(df_genel['NUMBER_OF_VEHICLES'].quantile(0.05))
max_arac = float(df_genel['NUMBER_OF_VEHICLES'].quantile(0.95))

print("\n3. ADIM: Referans tablosu üretiliyor ve src/ klasörüne kaydediliyor...")
ref_tablo = df_genel.groupby(['gun_kod', 'saat'])['NUMBER_OF_VEHICLES'].mean().reset_index()

arayuz_paket = {
    'ref_tablo': ref_tablo,
    'min_arac': min_arac,
    'max_arac': max_arac
}
# Yol '../src/' olarak güncellendi
with open('../src/4yillik_referans.pkl', 'wb') as f:
    pickle.dump(arayuz_paket, f)

print("\n4. ADIM: XGBoost Modeli eğitiliyor ve src/ klasörüne kaydediliyor...")
X = df_genel[['saat', 'gun_kod', 'hafta_sonu', 'bir_saat_once']]
y = df_genel['NUMBER_OF_VEHICLES']

model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)
model.fit(X, y)

# Yol '../src/' olarak güncellendi
with open('../src/trafik_modeli_4yil.pkl', 'wb') as dosya:
    pickle.dump(model, dosya)

print("\n[TAMAMLANDI] Yeni pkl dosyaları src/ klasörüne tıkır tıkır yazıldı!")

1. ADIM: data/ klasöründeki tüm CSV dosyaları okunuyor...

[BİLGİ] Filtrelemeden sonra toplam temiz satır sayısı: 39660

2. ADIM: Özellikler çıkarılıyor...

3. ADIM: Referans tablosu üretiliyor ve src/ klasörüne kaydediliyor...

4. ADIM: XGBoost Modeli eğitiliyor ve src/ klasörüne kaydediliyor...

[TAMAMLANDI] Yeni pkl dosyaları src/ klasörüne tıkır tıkır yazıldı!
